# Hybrid Bio+Clinical BERT with CNN for Drug Review Sentiment Analysis

This notebook compares 5 different NLP models for sentiment analysis on drug reviews:

1. **Baseline BERT (Feature Extraction)**: Frozen BERT + MLP
2. **CNN-Word2Vec**: Convolutional model with trainable embeddings
3. **BERT Base (Fine-Tuned)**: Fine-tuning of the entire model
4. **Bio+Clinical BERT (Fine-Tuned)**: Fine-tuning of the medical domain model
5. **Hybrid Bio+Clinical BERT + CNN**: The proposed model combining BERT embeddings with CNN

## 1. Configuration & Setup

In [ ]:
# ==============================================================================
# CONFIGURATION - Modify these settings as needed
# ==============================================================================

class Config:
    """Central configuration for all experiments."""
    
    # --- Paths ---
    DATA_DIR = "../data"
    MODEL_CACHE_DIR = "../pretrained_models"
    CHECKPOINT_DIR = "../checkpoints"
    RESULTS_DIR = "../experiment_results"
    
    # --- Model Paths ---
    BERT_MODEL_NAME = "bert-base-cased"
    BIO_CLINICAL_MODEL_NAME = "Bio_ClinicalBERT"
    
    # --- Training Hyperparameters ---
    MAX_LEN = 128
    BATCH_SIZE = 16  # Reduce to 8 if OOM errors occur
    N_FOLDS = 3
    EPOCHS = 4
    LEARNING_RATE = 2e-5
    
    # --- Model Architecture ---
    HIDDEN_DIM = 100
    DROPOUT = 0.3
    N_CLASSES = 3
    BERT_HIDDEN_SIZE = 768
    
    # --- Data Settings ---
    TEST_SIZE = 0.2
    RANDOM_STATE = 42
    
    # --- Sampling (set to None to use full dataset) ---
    SAMPLE_SIZE = None  # e.g., 10000 for quick testing
    
    @classmethod
    def bert_path(cls):
        return f"{cls.MODEL_CACHE_DIR}/{cls.BERT_MODEL_NAME}"
    
    @classmethod
    def bio_clinical_path(cls):
        return f"{cls.MODEL_CACHE_DIR}/{cls.BIO_CLINICAL_MODEL_NAME}"

config = Config()

In [ ]:
# ==============================================================================
# IMPORTS & ENVIRONMENT SETUP
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support, 
    accuracy_score, 
    confusion_matrix, 
    roc_curve, 
    auc,
    classification_report
)
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Create directories
for dir_path in [config.DATA_DIR, config.MODEL_CACHE_DIR, config.CHECKPOINT_DIR, config.RESULTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Dictionary to store results
ALL_MODEL_RESULTS = {}

## 2. Data Loading & Preprocessing

In [ ]:
# ==============================================================================
# DATA LOADING
# ==============================================================================

def bin_rating(rating):
    """Convert numeric rating to sentiment class."""
    try:
        r = float(rating)
    except (ValueError, TypeError):
        return None
    
    if r <= 4:
        return 0  # Negative
    elif 5 <= r <= 8:
        return 1  # Neutral
    else:  # r >= 9
        return 2  # Positive


def load_data(config):
    """Load and preprocess drug review dataset."""
    print("Loading data...")
    
    train_path = os.path.join(config.DATA_DIR, "drugsComTrain_raw.csv")
    test_path = os.path.join(config.DATA_DIR, "drugsComTest_raw.csv")
    
    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        raise FileNotFoundError(
            f"Data files not found in {config.DATA_DIR}.\n"
            "Please run: python scripts/download_data.py"
        )
    
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    df_all = pd.concat([df_train, df_test], ignore_index=True)
    
    print(f"Total samples loaded: {len(df_all)}")
    
    # Preprocessing
    df_all['label'] = df_all['rating'].apply(bin_rating)
    df_all = df_all.rename(columns={'review': 'text'})
    df_all = df_all.dropna(subset=['text', 'label'])
    df_all['label'] = df_all['label'].astype(int)
    
    # Optional sampling for quick experiments
    if config.SAMPLE_SIZE is not None and len(df_all) > config.SAMPLE_SIZE:
        df_all = df_all.sample(config.SAMPLE_SIZE, random_state=config.RANDOM_STATE)
        print(f"Sampled to {config.SAMPLE_SIZE} samples for quick testing.")
    
    # Train/Test split
    train_val_df, holdout_test_df = train_test_split(
        df_all,
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE,
        stratify=df_all['label']
    )
    
    print(f"Train/CV samples: {len(train_val_df)}")
    print(f"Hold-out test samples: {len(holdout_test_df)}")
    
    return train_val_df.reset_index(drop=True), holdout_test_df.reset_index(drop=True)


# Load data
train_val_df, holdout_test_df = load_data(config)

In [ ]:
# ==============================================================================
# DATASET CLASS
# ==============================================================================

class DrugReviewDataset(Dataset):
    """PyTorch Dataset for drug reviews."""
    
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

## 3. Data Visualization

In [ ]:
# ==============================================================================
# DATA VISUALIZATION
# ==============================================================================

LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL_ORDER = ['Negative', 'Neutral', 'Positive']

def plot_label_distribution(df, title="Data Distribution", ax=None):
    """Plot distribution of sentiment labels."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    
    df_viz = df.copy()
    df_viz['label_name'] = df_viz['label'].map(LABEL_MAP)
    
    sns.countplot(x='label_name', data=df_viz, order=LABEL_ORDER, palette='viridis', ax=ax)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    
    # Add count labels on bars
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10)


# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_label_distribution(train_val_df, "Train/CV Set Distribution", axes[0])
plot_label_distribution(holdout_test_df, "Hold-out Test Set Distribution", axes[1])

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "data_distribution.png"), dpi=150)
plt.show()

# Print class distribution
print("\nClass Distribution (Train/CV):")
print(train_val_df['label'].value_counts().sort_index().to_frame().rename(
    index=LABEL_MAP, columns={'label': 'count'}))

## 4. Model Architectures

In [ ]:
# ==============================================================================
# MODEL 1: BASELINE BERT (FROZEN) + MLP
# ==============================================================================

class BertBaseline(nn.Module):
    """Frozen BERT with MLP classifier (Feature Extraction)."""
    
    def __init__(self, model_path, n_classes=3, hidden_dim=100):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        
        # Freeze all BERT parameters
        for param in self.bert.parameters():
            param.requires_grad = False
        
        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask)
        pooled_output = outputs.pooler_output  # [CLS] token representation
        return self.classifier(pooled_output)

In [ ]:
# ==============================================================================
# MODEL 2: CNN TEXT CLASSIFIER
# ==============================================================================

class CNNTextClassifier(nn.Module):
    """CNN-based text classifier with multiple filter sizes."""
    
    def __init__(self, vocab_size, embed_dim=300, n_classes=3, 
                 filter_sizes=[1, 2, 3, 4, 5], n_filters=100, 
                 dropout=0.5, pretrained_weights=None):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        if pretrained_weights is not None:
            self.embedding.load_state_dict({'weight': torch.tensor(pretrained_weights)})
            self.embedding.weight.requires_grad = False
        
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, n_filters, kernel_size=k)
            for k in filter_sizes
        ])
        
        self.fc = nn.Linear(len(filter_sizes) * n_filters, 100)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(100, n_classes)
    
    def forward(self, input_ids, attention_mask=None):
        # Embedding: (batch, seq_len, embed_dim)
        x = self.embedding(input_ids)
        # Transpose for Conv1d: (batch, embed_dim, seq_len)
        x = x.permute(0, 2, 1)
        
        # Apply convolutions and max pooling
        conv_outputs = []
        for conv in self.convs:
            conv_out = F.relu(conv(x))
            pooled = F.max_pool1d(conv_out, conv_out.size(2)).squeeze(2)
            conv_outputs.append(pooled)
        
        # Concatenate all filter outputs
        x = torch.cat(conv_outputs, dim=1)
        x = F.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)

In [ ]:
# ==============================================================================
# MODEL 3 & 4: FINE-TUNED BERT
# ==============================================================================

class BertFineTune(nn.Module):
    """Fine-tuned BERT with unfrozen top layers."""
    
    def __init__(self, model_path, n_classes=3, hidden_dim=100, 
                 n_unfreeze_layers=4, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        
        # Freeze all parameters first
        for param in self.bert.parameters():
            param.requires_grad = False
        
        # Unfreeze last n layers
        for layer in self.bert.encoder.layer[-n_unfreeze_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
        
        # Unfreeze pooler
        if hasattr(self.bert, 'pooler') and self.bert.pooler is not None:
            for param in self.bert.pooler.parameters():
                param.requires_grad = True
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask)
        return self.classifier(outputs.pooler_output)

In [ ]:
# ==============================================================================
# MODEL 5: HYBRID BIO+CLINICAL BERT + CNN (PROPOSED MODEL)
# ==============================================================================

class HybridBioClinicalBertCNN(nn.Module):
    """
    Hybrid model combining Bio+Clinical BERT embeddings with CNN.
    
    This is the proposed model that leverages:
    - Domain-specific BERT embeddings (Bio+Clinical BERT)
    - Multi-scale CNN feature extraction
    """
    
    def __init__(self, model_path, n_classes=3, n_filters=100, 
                 filter_sizes=[3, 4, 5], dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        
        # CNN layers with different kernel sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(768, n_filters, kernel_size=k, padding=k//2)
            for k in filter_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(filter_sizes), n_classes)
    
    def forward(self, input_ids, attention_mask):
        # Get BERT hidden states
        outputs = self.bert(input_ids, attention_mask)
        # Use last hidden state: (batch, seq_len, 768)
        hidden_states = outputs.last_hidden_state
        # Transpose for Conv1d: (batch, 768, seq_len)
        hidden_states = hidden_states.permute(0, 2, 1)
        
        # Apply CNN layers and global max pooling
        conv_outputs = []
        for conv in self.convs:
            conv_out = F.relu(conv(hidden_states))
            pooled = F.max_pool1d(conv_out, conv_out.size(2)).squeeze(2)
            conv_outputs.append(pooled)
        
        # Concatenate and classify
        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

## 5. Training Engine

In [ ]:
# ==============================================================================
# TRAINING ENGINE WITH K-FOLD CROSS-VALIDATION
# ==============================================================================

def train_and_evaluate(
    model_name: str,
    model: nn.Module,
    tokenizer,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    config: Config,
    results_dict: dict
):
    """
    Train model using K-Fold CV and evaluate on hold-out test set.
    
    Args:
        model_name: Name for saving checkpoints and results
        model: PyTorch model to train
        tokenizer: Tokenizer for text encoding
        train_df: Training/validation DataFrame
        test_df: Hold-out test DataFrame
        config: Configuration object
        results_dict: Dictionary to store results
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    checkpoint_path = os.path.join(config.CHECKPOINT_DIR, f"{model_name}_best.pth")
    skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
    
    best_val_acc = 0.0
    
    # Check for existing checkpoint
    if os.path.exists(checkpoint_path):
        print(f"Loading existing checkpoint: {checkpoint_path}")
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        model.to(device)
    else:
        # K-Fold Training
        for fold, (train_idx, val_idx) in enumerate(skf.split(train_df['text'], train_df['label'])):
            print(f"\n--- Fold {fold + 1}/{config.N_FOLDS} ---")
            
            # Split data
            fold_train = train_df.iloc[train_idx]
            fold_val = train_df.iloc[val_idx]
            
            # Balance training data using oversampling
            ros = RandomOverSampler(random_state=config.RANDOM_STATE)
            X_resampled, y_resampled = ros.fit_resample(
                fold_train['text'].values.reshape(-1, 1),
                fold_train['label'].values
            )
            fold_train_balanced = pd.DataFrame({
                'text': X_resampled.flatten(),
                'label': y_resampled
            })
            
            # Create DataLoaders
            train_dataset = DrugReviewDataset(
                fold_train_balanced['text'].values,
                fold_train_balanced['label'].values,
                tokenizer, config.MAX_LEN
            )
            val_dataset = DrugReviewDataset(
                fold_val['text'].values,
                fold_val['label'].values,
                tokenizer, config.MAX_LEN
            )
            
            train_loader = DataLoader(
                train_dataset, batch_size=config.BATCH_SIZE, 
                shuffle=True, pin_memory=True, num_workers=0
            )
            val_loader = DataLoader(
                val_dataset, batch_size=config.BATCH_SIZE,
                pin_memory=True, num_workers=0
            )
            
            # Setup training
            model.to(device)
            optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE)
            criterion = nn.CrossEntropyLoss()
            
            # Training loop
            for epoch in range(config.EPOCHS):
                model.train()
                total_loss = 0
                
                progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.EPOCHS}", leave=False)
                for batch in progress_bar:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['label'].to(device)
                    
                    optimizer.zero_grad()
                    outputs = model(input_ids, attention_mask)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    
                    total_loss += loss.item()
                    progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
                
                # Validation
                model.eval()
                correct = 0
                total = 0
                
                with torch.no_grad():
                    for batch in val_loader:
                        input_ids = batch['input_ids'].to(device)
                        attention_mask = batch['attention_mask'].to(device)
                        labels = batch['label'].to(device)
                        
                        outputs = model(input_ids, attention_mask)
                        _, preds = torch.max(outputs, dim=1)
                        correct += (preds == labels).sum().item()
                        total += labels.size(0)
                
                val_acc = correct / total
                avg_loss = total_loss / len(train_loader)
                print(f"  Epoch {epoch + 1}: Loss = {avg_loss:.4f}, Val Acc = {val_acc:.4f}")
                
                # Save best model
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    torch.save(model.state_dict(), checkpoint_path)
                    print(f"  -> New best model saved (Val Acc: {val_acc:.4f})")
    
    # Final evaluation on hold-out test set
    print(f"\nEvaluating {model_name} on hold-out test set...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    
    test_dataset = DrugReviewDataset(
        test_df['text'].values,
        test_df['label'].values,
        tokenizer, config.MAX_LEN
    )
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, pin_memory=True)
    
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, attention_mask)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Store results
    results_dict[model_name] = {
        'y_true': np.array(all_labels),
        'y_pred': np.array(all_preds),
        'y_probs': np.array(all_probs)
    }
    
    # Print quick summary
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc:.4f}")
    
    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()

## 6. Run All Experiments

In [ ]:
# ==============================================================================
# LOAD TOKENIZERS
# ==============================================================================

print("Loading tokenizers...")
print(f"  BERT path: {config.bert_path()}")
print(f"  Bio+Clinical BERT path: {config.bio_clinical_path()}")

bert_tokenizer = AutoTokenizer.from_pretrained(config.bert_path(), local_files_only=True)
bio_clinical_tokenizer = AutoTokenizer.from_pretrained(config.bio_clinical_path(), local_files_only=True)

print("Tokenizers loaded successfully!")

In [ ]:
# ==============================================================================
# EXPERIMENT 1: BERT BASELINE (FROZEN)
# ==============================================================================

model1 = BertBaseline(
    model_path=config.bert_path(),
    n_classes=config.N_CLASSES,
    hidden_dim=config.HIDDEN_DIM
)

train_and_evaluate(
    model_name="1_BERT_Baseline",
    model=model1,
    tokenizer=bert_tokenizer,
    train_df=train_val_df,
    test_df=holdout_test_df,
    config=config,
    results_dict=ALL_MODEL_RESULTS
)

In [ ]:
# ==============================================================================
# EXPERIMENT 2: CNN TEXT CLASSIFIER
# ==============================================================================

model2 = CNNTextClassifier(
    vocab_size=bert_tokenizer.vocab_size,
    embed_dim=300,
    n_classes=config.N_CLASSES
)

train_and_evaluate(
    model_name="2_CNN_Model",
    model=model2,
    tokenizer=bert_tokenizer,
    train_df=train_val_df,
    test_df=holdout_test_df,
    config=config,
    results_dict=ALL_MODEL_RESULTS
)

In [ ]:
# ==============================================================================
# EXPERIMENT 3: BERT FINE-TUNED
# ==============================================================================

model3 = BertFineTune(
    model_path=config.bert_path(),
    n_classes=config.N_CLASSES,
    hidden_dim=config.HIDDEN_DIM
)

train_and_evaluate(
    model_name="3_BERT_FineTuned",
    model=model3,
    tokenizer=bert_tokenizer,
    train_df=train_val_df,
    test_df=holdout_test_df,
    config=config,
    results_dict=ALL_MODEL_RESULTS
)

In [ ]:
# ==============================================================================
# EXPERIMENT 4: BIO+CLINICAL BERT FINE-TUNED
# ==============================================================================

model4 = BertFineTune(
    model_path=config.bio_clinical_path(),
    n_classes=config.N_CLASSES,
    hidden_dim=config.HIDDEN_DIM
)

train_and_evaluate(
    model_name="4_BioClinicalBERT_FineTuned",
    model=model4,
    tokenizer=bio_clinical_tokenizer,
    train_df=train_val_df,
    test_df=holdout_test_df,
    config=config,
    results_dict=ALL_MODEL_RESULTS
)

In [ ]:
# ==============================================================================
# EXPERIMENT 5: HYBRID BIO+CLINICAL BERT + CNN (PROPOSED MODEL)
# ==============================================================================

model5 = HybridBioClinicalBertCNN(
    model_path=config.bio_clinical_path(),
    n_classes=config.N_CLASSES,
    dropout=config.DROPOUT
)

train_and_evaluate(
    model_name="5_Hybrid_BioClinicalBERT_CNN",
    model=model5,
    tokenizer=bio_clinical_tokenizer,
    train_df=train_val_df,
    test_df=holdout_test_df,
    config=config,
    results_dict=ALL_MODEL_RESULTS
)

print("\n" + "="*60)
print("All experiments completed!")
print("="*60)

## 7. Results Visualization & Analysis

In [ ]:
# ==============================================================================
# ROC CURVE COMPARISON
# ==============================================================================

def plot_roc_curves(results_dict, save_path=None):
    """Plot ROC curves for all models."""
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(results_dict)))
    
    for idx, (name, results) in enumerate(results_dict.items()):
        y_true = results['y_true']
        y_probs = results['y_probs']
        
        # Binarize labels for multi-class ROC
        y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
        n_classes = 3
        
        # Compute ROC for each class
        fpr = {}
        tpr = {}
        roc_auc = {}
        
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
        
        # Compute macro-average ROC
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= n_classes
        
        macro_auc = auc(all_fpr, mean_tpr)
        
        # Plot
        plt.plot(all_fpr, mean_tpr, color=colors[idx], lw=2,
                 label=f'{name} (AUC = {macro_auc:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves Comparison (Macro-Average)', fontsize=14)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_roc_curves(
    ALL_MODEL_RESULTS, 
    save_path=os.path.join(config.RESULTS_DIR, "roc_curves.png")
)

In [ ]:
# ==============================================================================
# CONFUSION MATRICES
# ==============================================================================

def plot_confusion_matrices(results_dict, save_path=None):
    """Plot confusion matrices for all models."""
    n_models = len(results_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
    
    if n_models == 1:
        axes = [axes]
    
    class_names = ['Neg', 'Neu', 'Pos']
    
    for idx, (name, results) in enumerate(results_dict.items()):
        cm = confusion_matrix(results['y_true'], results['y_pred'])
        
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[idx], cbar=False
        )
        axes[idx].set_title(name.replace('_', '\n'), fontsize=10)
        axes[idx].set_xlabel('Predicted')
        if idx == 0:
            axes[idx].set_ylabel('True')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_confusion_matrices(
    ALL_MODEL_RESULTS,
    save_path=os.path.join(config.RESULTS_DIR, "confusion_matrices.png")
)

In [ ]:
# ==============================================================================
# PERFORMANCE SUMMARY TABLE
# ==============================================================================

def generate_performance_summary(results_dict):
    """Generate comprehensive performance summary."""
    summary_data = []
    
    for name, results in results_dict.items():
        y_true = results['y_true']
        y_pred = results['y_pred']
        y_probs = results['y_probs']
        
        # Basic metrics
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro'
        )
        
        # Compute AUC
        y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
        auc_scores = []
        for i in range(3):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            auc_scores.append(auc(fpr, tpr))
        macro_auc = np.mean(auc_scores)
        
        summary_data.append({
            'Model': name,
            'Accuracy': acc,
            'Macro Precision': precision,
            'Macro Recall': recall,
            'Macro F1': f1,
            'Macro AUC': macro_auc
        })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary = df_summary.sort_values(by='Macro F1', ascending=False)
    
    return df_summary


# Generate and display summary
df_summary = generate_performance_summary(ALL_MODEL_RESULTS)

print("\n" + "="*80)
print("FINAL PERFORMANCE SUMMARY")
print("="*80)

# Format for display
df_display = df_summary.copy()
for col in ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Macro AUC']:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")

print(df_display.to_string(index=False))

# Save to CSV
csv_path = os.path.join(config.RESULTS_DIR, "performance_summary.csv")
df_summary.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

In [ ]:
# ==============================================================================
# BAR CHART COMPARISON
# ==============================================================================

def plot_metrics_comparison(df_summary, save_path=None):
    """Plot bar chart comparing all metrics across models."""
    metrics = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Macro AUC']
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(df_summary))
    width = 0.15
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(metrics)))
    
    for i, metric in enumerate(metrics):
        offset = (i - len(metrics)/2 + 0.5) * width
        bars = ax.bar(x + offset, df_summary[metric], width, 
                      label=metric, color=colors[i])
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Model Performance Comparison', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', '\n') for m in df_summary['Model']], 
                       rotation=0, fontsize=9)
    ax.legend(loc='lower right', fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_metrics_comparison(
    df_summary,
    save_path=os.path.join(config.RESULTS_DIR, "metrics_comparison.png")
)

In [ ]:
# ==============================================================================
# DETAILED CLASSIFICATION REPORTS
# ==============================================================================

print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

target_names = ['Negative', 'Neutral', 'Positive']

for name, results in ALL_MODEL_RESULTS.items():
    print(f"\n--- {name} ---")
    print(classification_report(
        results['y_true'], 
        results['y_pred'],
        target_names=target_names
    ))

## 8. Conclusion

This notebook compared 5 different approaches for drug review sentiment analysis:

1. **BERT Baseline**: Feature extraction with frozen BERT
2. **CNN Model**: Traditional CNN with trainable embeddings
3. **BERT Fine-Tuned**: Standard BERT with partial fine-tuning
4. **Bio+Clinical BERT**: Domain-specific BERT for medical text
5. **Hybrid Bio+Clinical BERT + CNN**: Proposed model combining domain expertise with CNN feature extraction

The results demonstrate the effectiveness of domain-specific pre-training and the benefits of combining BERT representations with CNN architectures for medical sentiment analysis.